# PySpark practice 1: execution model

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-practice-01-execution-model.ipynb)

12 exercises on how Spark executes your code: shuffles, partitions, plans, pushdown, AQE, caching and lineage. Each exercise names the [reference entries](pyspark-reference-01-execution-model.ipynb) it practises, shows its inputs and the exact expected output, and hides a reference solution you can reveal when you are done or stuck.

*Last reviewed: November 2026, against Apache Spark 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

A local session, the shared `customers`, `orders`, `order_items`, `products` and `events` tables (customer 4 has no region, customers 7 and 8 never ordered, `SKU-4014` has no product row, one event is an exact retry), and the helpers from [PySpark reference 1](pyspark-reference-01-execution-model.ipynb#setup-and-sample-data). Exercises that need other data create it in their input cell.

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import Window
from pyspark.sql import functions as F

customers = spark.createDataFrame([
    (1, "Acme Corp", "North", date(2024, 1, 3)),
    (2, "Bright Retail", "North", date(2024, 2, 11)),
    (3, "Cyan Traders", "South", date(2024, 3, 15)),
    (4, "Delta Grocers", None, date(2024, 5, 2)),  # region never arrived from the CRM
    (5, "Echo Supplies", "West", date(2024, 6, 20)),
    (6, "Foxtrot Foods", "West", date(2024, 8, 1)),
    (7, "Gamma Sports", "East", date(2024, 9, 14)),  # 7 and 8 have never ordered
    (8, "Hotel Homeware", "East", date(2024, 11, 5)),
], "customer_id int, name string, region string, signup_date date")

orders = spark.createDataFrame([
    (1, 1, "paid", datetime(2026, 1, 5, 9, 15), D("120.00")),
    (2, 1, "shipped", datetime(2026, 1, 5, 10, 5), D("84.50")),
    (3, 1, "returned", datetime(2026, 1, 5, 11, 40), D("30.00")),
    (4, 2, "paid", datetime(2026, 1, 6, 15, 20), D("260.00")),
    (5, 2, "placed", datetime(2026, 1, 7, 8, 5), D("55.25")),
    (6, 3, "shipped", datetime(2026, 1, 7, 17, 45), D("149.00")),
    (7, 3, "paid", datetime(2026, 1, 9, 12, 10), D("310.00")),
    (8, 4, "returned", datetime(2026, 1, 10, 19, 30), D("89.90")),
    (9, 4, "shipped", datetime(2026, 1, 12, 8, 55), D("1180.00")),
    (10, 5, "paid", datetime(2026, 1, 12, 13, 0), D("74.00")),
    (11, 6, "placed", datetime(2026, 1, 13, 10, 25), D("210.00")),
    (12, 6, "shipped", datetime(2026, 1, 14, 0, 30), D("96.40")),  # just after midnight UTC
], "order_id int, customer_id int, status string, order_ts timestamp, amount decimal(12,2)")

order_items = spark.createDataFrame([
    (1, "SKU-4001", 2, D("12.50")), (1, "SKU-4007", 1, D("95.00")),
    (2, "SKU-4002", 1, D("34.50")), (2, "SKU-4009", 2, D("25.00")),
    (3, "SKU-4002", 1, D("30.00")),
    (4, "SKU-4003", 1, D("149.00")), (4, "SKU-4004", 2, D("55.50")),
    (5, "SKU-4005", 1, D("55.25")),
    (6, "SKU-4003", 1, D("149.00")),
    (7, "SKU-4006", 1, D("189.00")), (7, "SKU-4008", 1, D("121.00")),
    (8, "SKU-4010", 1, D("89.90")),
    (9, "SKU-4011", 20, D("59.00")),
    (10, "SKU-4012", 2, D("37.00")),
    (11, "SKU-4013", 1, D("210.00")),
    (12, "SKU-4014", 1, D("41.40")), (12, "SKU-4005", 1, D("55.00")),
], "order_id int, sku string, qty int, unit_price decimal(8,2)")

products = spark.createDataFrame([
    ("SKU-4001", "Kitchen", D("12.50")), ("SKU-4002", "Kitchen", D("34.50")),
    ("SKU-4003", "Garden", D("149.00")), ("SKU-4004", "Garden", D("55.50")),
    ("SKU-4005", "Pantry", D("55.25")), ("SKU-4006", "Outdoor", D("189.00")),
    ("SKU-4007", "Kitchen", D("95.00")), ("SKU-4008", "Outdoor", D("121.00")),
    ("SKU-4009", "Pantry", D("25.00")), ("SKU-4010", "Garden", D("89.90")),
    ("SKU-4011", "Pantry", D("59.00")), ("SKU-4012", "Kitchen", D("37.00")),
    ("SKU-4013", "Outdoor", D("210.00")),  # SKU-4014 is missing
], "sku string, category string, list_price decimal(8,2)")

events = spark.createDataFrame([
    (1, "view", datetime(2026, 1, 6, 9, 0)), (1, "add_to_cart", datetime(2026, 1, 6, 9, 2)),
    (1, "purchase", datetime(2026, 1, 6, 9, 10)), (2, "view", datetime(2026, 1, 7, 10, 0)),
    (2, "purchase", datetime(2026, 1, 7, 10, 40)), (3, "view", datetime(2026, 1, 6, 11, 15)),
    (3, "view", datetime(2026, 1, 8, 11, 20)), (4, "add_to_cart", datetime(2026, 1, 9, 20, 5)),
    (4, "purchase", datetime(2026, 1, 10, 8, 30)), (5, "view", datetime(2026, 1, 11, 14, 45)),
    (5, "view", datetime(2026, 1, 12, 9, 0)), (5, "purchase", datetime(2026, 1, 12, 12, 30)),
    (5, "view", datetime(2026, 1, 12, 9, 0)),  # an SDK retry committed this event twice
], "user_id int, event string, ts timestamp")

import contextlib
import io
import logging
import re


def plan_lines(df, mode="simple"):
    """The lines df.explain(mode) prints."""
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        df.explain(mode)
    return buffer.getvalue().splitlines()


def final_plan(df):
    """Run df, then return the plan Spark actually used (AQE's final plan when there is one)."""
    df.collect()
    text = "\n".join(plan_lines(df))
    if "== Final Plan ==" in text:
        text = text.split("== Final Plan ==")[1].split("== Initial Plan ==")[0]
    return text


@contextlib.contextmanager
def expected_error():
    """Run a block that is meant to fail and print the error class instead of a stack trace."""
    spark.sparkContext.setLogLevel("OFF")
    for logger in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
        logging.getLogger(logger).setLevel(logging.CRITICAL)
    try:
        yield
    except Exception as error:
        print(type(error).__name__ + ":", str(error).split(".")[0][:100])
    finally:
        spark.sparkContext.setLogLevel("ERROR")

## Exercises

4 easy, 5 medium, 3 hard. Write your answer in the empty cell under each exercise as a DataFrame called `result`, run it, and compare it with the expected output (columns, values and row order).

| # | exercise | difficulty | practises |
| --- | --- | --- | --- |
| 1 | [Count the shuffles](#exercise-1-count-the-shuffles) | Easy | [Narrow and wide transformations](pyspark-reference-01-execution-model.ipynb#narrow-and-wide-transformations) |
| 2 | [Partitions after repartition and coalesce](#exercise-2-partitions-after-repartition-and-coalesce) | Easy | [`repartition` and `coalesce`](pyspark-reference-01-execution-model.ipynb#repartition-and-coalesce) |
| 3 | [Which keys share a partition](#exercise-3-which-keys-share-a-partition) | Medium | [Partition count: reading, shuffling and AQE](pyspark-reference-01-execution-model.ipynb#partition-count-reading-shuffling-and-aqe) |
| 4 | [Find what was pushed into the scan](#exercise-4-find-what-was-pushed-into-the-scan) | Medium | [Predicate and projection pushdown](pyspark-reference-01-execution-model.ipynb#predicate-and-projection-pushdown) |
| 5 | [AQE coalesces shuffle partitions](#exercise-5-aqe-coalesces-shuffle-partitions) | Medium | [Adaptive Query Execution (AQE)](pyspark-reference-01-execution-model.ipynb#adaptive-query-execution-aqe) |
| 6 | [Cache a DataFrame used by several outputs](#exercise-6-cache-a-dataframe-used-by-several-outputs) | Easy | [`cache`, `persist` and `unpersist`](pyspark-reference-01-execution-model.ipynb#cache-persist-and-unpersist) |
| 7 | [Lazy evaluation and where errors surface](#exercise-7-lazy-evaluation-and-where-errors-surface) | Medium | [Transformations and actions](pyspark-reference-01-execution-model.ipynb#transformations-and-actions) |
| 8 | [Cut a long lineage](#exercise-8-cut-a-long-lineage) | Medium | [`checkpoint` and `localCheckpoint`](pyspark-reference-01-execution-model.ipynb#checkpoint-and-localcheckpoint) |
| 9 | [Spot the two-step aggregation](#exercise-9-spot-the-two-step-aggregation) | Easy | [Physical operators you will see](pyspark-reference-01-execution-model.ipynb#physical-operators-you-will-see) |
| 10 | [Rows per partition after a skewed repartition](#exercise-10-rows-per-partition-after-a-skewed-repartition) | Hard | [`repartition` and `coalesce`](pyspark-reference-01-execution-model.ipynb#repartition-and-coalesce) |
| 11 | [Partitions when reading many small files](#exercise-11-partitions-when-reading-many-small-files) | Hard | [Partition count: reading, shuffling and AQE](pyspark-reference-01-execution-model.ipynb#partition-count-reading-shuffling-and-aqe) |
| 12 | [Count jobs and stages of an action](#exercise-12-count-jobs-and-stages-of-an-action) | Hard | [How an action becomes jobs, stages and tasks](pyspark-reference-01-execution-model.ipynb#how-an-action-becomes-jobs-stages-and-tasks) |

## Exercise 1: Count the shuffles

**Difficulty:** Easy · about 5 min · **Practises:** [Narrow and wide transformations](pyspark-reference-01-execution-model.ipynb#narrow-and-wide-transformations)

*Interviewers ask which operations shuffle; the plan answers it for certain.*

For each query in `queries`, return `query` and `shuffles` (the number of `Exchange` operators in its plan). Keep the order of `queries`.

In [ ]:
queries = {
    "filter_select": orders.where("amount > 100").select("order_id"),
    "group_count": orders.groupBy("status").count(),
    "distinct_status": orders.select("status").distinct(),
    "sort_amount": orders.orderBy("amount"),
    "union": orders.union(orders),
}
for name in queries:
    print(name)

**Expected output**

| query | shuffles |
| --- | --- |
| filter_select | 0 |
| group_count | 1 |
| distinct_status | 1 |
| sort_amount | 1 |
| union | 0 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
rows = [(name, sum("Exchange" in line for line in plan_lines(df))) for name, df in queries.items()]
result = spark.createDataFrame(rows, "query string, shuffles int")
result.show()

## Exercise 2: Partitions after repartition and coalesce

**Difficulty:** Easy · about 5 min · **Practises:** [`repartition` and `coalesce`](pyspark-reference-01-execution-model.ipynb#repartition-and-coalesce)

*"repartition vs coalesce" is a standard interview question; know what each can and cannot do.*

Starting from `base` (8 partitions), return `operation` and `partitions` for: `repartition(3)`, `coalesce(3)`, `coalesce(20)` and `repartition(20)`, in that order.

In [ ]:
base = spark.range(100, numPartitions=8)
print(base.rdd.getNumPartitions())

**Expected output**

| operation | partitions |
| --- | --- |
| repartition(3) | 3 |
| coalesce(3) | 3 |
| coalesce(20) | 8 |
| repartition(20) | 20 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
ops = {"repartition(3)": base.repartition(3), "coalesce(3)": base.coalesce(3),
       "coalesce(20)": base.coalesce(20), "repartition(20)": base.repartition(20)}
result = spark.createDataFrame([(k, v.rdd.getNumPartitions()) for k, v in ops.items()], "operation string, partitions int")
result.show()

## Exercise 3: Which keys share a partition

**Difficulty:** Medium · about 10 min · **Practises:** [Partition count: reading, shuffling and AQE](pyspark-reference-01-execution-model.ipynb#partition-count-reading-shuffling-and-aqe)

*Hash partitioning guarantees co-location of a key, which is what joins and aggregations rely on.*

Repartition `orders` into 2 partitions by `customer_id`. Return `partition` and `customers` (the sorted array of distinct customer ids in it) as a string joined by `,`, ordered by `partition`.

In [ ]:
orders.select("order_id", "customer_id").show()

**Expected output**

| partition | customers |
| --- | --- |
| 0 | 2,4,5 |
| 1 | 1,3,6 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
spread = orders.repartition(2, "customer_id").select("customer_id", F.spark_partition_id().alias("partition"))
result = (
    spread.groupBy("partition")
    .agg(F.array_join(F.sort_array(F.collect_set(F.col("customer_id").cast("string"))), ",").alias("customers"))
    .orderBy("partition")
)
result.show()

## Exercise 4: Find what was pushed into the scan

**Difficulty:** Medium · about 10 min · **Practises:** [Predicate and projection pushdown](pyspark-reference-01-execution-model.ipynb#predicate-and-projection-pushdown)

*Checking pushdown is the first step when a filter on a big table is slow.*

Write `orders` as Parquet partitioned by `status` under `WORK_DIR`, read it back, filter `status = 'shipped' AND amount >= 100` and select `order_id`. From the formatted plan, return `property` and `present` (true/false) for `partition_filter_on_status` (status appears in `PartitionFilters`), `pushed_filter_on_amount` (amount appears in `PushedFilters`) and `read_schema_has_customer_id` (customer_id appears in `ReadSchema`), in that order.

In [ ]:
target_path = f"{WORK_DIR}/practice_orders_by_status"
print(target_path)

**Expected output**

| property | present |
| --- | --- |
| partition_filter_on_status | true |
| pushed_filter_on_amount | true |
| read_schema_has_customer_id | false |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
orders.write.mode("overwrite").partitionBy("status").parquet(target_path)
scan = spark.read.parquet(target_path).where("status = 'shipped' AND amount >= 100").select("order_id")
text = "\n".join(plan_lines(scan, "formatted"))
partition_line = next(l for l in text.splitlines() if "PartitionFilters" in l)
pushed_line = next(l for l in text.splitlines() if "PushedFilters" in l)
schema_line = next(l for l in text.splitlines() if "ReadSchema" in l)
result = spark.createDataFrame([
    ("partition_filter_on_status", "status" in partition_line),
    ("pushed_filter_on_amount", "amount" in pushed_line),
    ("read_schema_has_customer_id", "customer_id" in schema_line),
], "property string, present boolean")
result.show()

## Exercise 5: AQE coalesces shuffle partitions

**Difficulty:** Medium · about 10 min · **Practises:** [Adaptive Query Execution (AQE)](pyspark-reference-01-execution-model.ipynb#adaptive-query-execution-aqe)

*Why does a job configured for 200 shuffle partitions run 1 task? AQE coalescing.*

With `spark.sql.shuffle.partitions` set to 50, run `orders.groupBy('status').count()` with AQE enabled and with `spark.sql.adaptive.enabled` set to false. Return `aqe` (true/false) and `partitions` (partition count of the result after it ran). Restore AQE afterwards.

In [ ]:
print(spark.conf.get("spark.sql.adaptive.enabled"), spark.conf.get("spark.sql.shuffle.partitions"))

**Expected output**

| aqe | partitions |
| --- | --- |
| true | 1 |
| false | 50 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
spark.conf.set("spark.sql.shuffle.partitions", "50")
rows = []
for aqe in [True, False]:
    spark.conf.set("spark.sql.adaptive.enabled", str(aqe).lower())
    grouped = orders.groupBy("status").count()
    grouped.collect()
    rows.append((aqe, grouped.rdd.getNumPartitions()))
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.shuffle.partitions", "4")
result = spark.createDataFrame(rows, "aqe boolean, partitions int")
result.show()

## Exercise 6: Cache a DataFrame used by several outputs

**Difficulty:** Easy · about 5 min · **Practises:** [`cache`, `persist` and `unpersist`](pyspark-reference-01-execution-model.ipynb#cache-persist-and-unpersist)

*Reusing an expensive intermediate result without recomputing it is the main reason to cache.*

Persist `enriched` (orders joined to customers) with `DISK_ONLY`, materialize it, and return one row with `is_cached`, `uses_disk`, `uses_memory` (from `storageLevel`) and `reads_cache` (whether a `groupBy('region')` on it reads `InMemoryTableScan`). Unpersist it at the end.

In [ ]:
enriched = orders.join(customers, "customer_id")

**Expected output**

| is_cached | uses_disk | uses_memory | reads_cache |
| --- | --- | --- | --- |
| true | true | false | true |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
from pyspark import StorageLevel

enriched.persist(StorageLevel.DISK_ONLY)
enriched.count()
level = enriched.storageLevel
reads_cache = any("InMemoryTableScan" in line for line in plan_lines(enriched.groupBy("region").count()))
result = spark.createDataFrame([(enriched.is_cached, level.useDisk, level.useMemory, reads_cache)],
                               "is_cached boolean, uses_disk boolean, uses_memory boolean, reads_cache boolean")
enriched.unpersist()
result.show()

## Exercise 7: Lazy evaluation and where errors surface

**Difficulty:** Medium · about 10 min · **Practises:** [Transformations and actions](pyspark-reference-01-execution-model.ipynb#transformations-and-actions)

*"My job failed on line 400 but the bug is on line 12": errors in transformations surface at the action.*

For each step, record whether it raised an exception: (1) defining `orders.select('nope')`, (2) defining `orders.withColumn('x', F.col('status').cast('int'))`, (3) calling `.count()` on the DataFrame from step 2, (4) calling `.collect()` on it. Return `step` and `raised`. Can you explain why steps 3 and 4 differ?

In [ ]:
print("steps: define_bad_column, define_bad_cast, count_bad_cast, collect_bad_cast")

**Expected output**

| step | raised |
| --- | --- |
| define_bad_column | true |
| define_bad_cast | false |
| count_bad_cast | false |
| collect_bad_cast | true |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
def raised(fn):
    spark.sparkContext.setLogLevel("OFF")
    for logger in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
        logging.getLogger(logger).setLevel(logging.CRITICAL)
    try:
        fn()
        return False
    except Exception:
        return True
    finally:
        spark.sparkContext.setLogLevel("ERROR")

bad_cast = orders.withColumn("x", F.col("status").cast("int"))
result = spark.createDataFrame([
    ("define_bad_column", raised(lambda: orders.select("nope"))),
    ("define_bad_cast", raised(lambda: orders.withColumn("x", F.col("status").cast("int")))),
    ("count_bad_cast", raised(lambda: bad_cast.count())),
    ("collect_bad_cast", raised(lambda: bad_cast.collect())),
], "step string, raised boolean")
result.show()

## Exercise 8: Cut a long lineage

**Difficulty:** Medium · about 10 min · **Practises:** [`checkpoint` and `localCheckpoint`](pyspark-reference-01-execution-model.ipynb#checkpoint-and-localcheckpoint)

*Iterative jobs that loop over withColumn grow plans until planning dominates run time.*

Apply `withColumn('amount', amount + 1)` to `orders` ten times. Return `stage` (`before`, `after`) and `project_operators`: the number of `Project` operators in the optimized logical plan section of `explain('extended')`, before and after `localCheckpoint()`.

In [ ]:
print(len(plan_lines(orders, "extended")))

**Expected output**

| stage | project_operators |
| --- | --- |
| before | 10 |
| after | 0 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
looped = orders
for _ in range(10):
    looped = looped.withColumn("amount", F.col("amount") + 1)

def projects(df):
    text = "\n".join(plan_lines(df, "extended"))
    analyzed = text.split("== Analyzed Logical Plan ==")[1].split("== Optimized Logical Plan ==")[0]
    return sum(line.strip().lstrip("+- :").startswith("Project") for line in analyzed.splitlines())

result = spark.createDataFrame([("before", projects(looped)), ("after", projects(looped.localCheckpoint()))],
                               "stage string, project_operators int")
result.show()

## Exercise 9: Spot the two-step aggregation

**Difficulty:** Easy · about 5 min · **Practises:** [Physical operators you will see](pyspark-reference-01-execution-model.ipynb#physical-operators-you-will-see)

*Partial aggregation is why groupBy shuffles far less data than the input size.*

For `orders.groupBy('customer_id').agg(F.sum('amount'))`, return one row with `partial_before_exchange` (true if a `partial_sum` appears in the plan) and `hash_aggregates` (the number of `HashAggregate` operators).

In [ ]:
print("orders.groupBy('customer_id').agg(F.sum('amount'))")

**Expected output**

| partial_before_exchange | hash_aggregates |
| --- | --- |
| true | 2 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
lines = plan_lines(orders.groupBy("customer_id").agg(F.sum("amount")))
result = spark.createDataFrame([(any("partial_sum" in l for l in lines), sum("HashAggregate" in l for l in lines))],
                               "partial_before_exchange boolean, hash_aggregates int")
result.show()

## Exercise 10: Rows per partition after a skewed repartition

**Difficulty:** Hard · about 15 min · **Practises:** [`repartition` and `coalesce`](pyspark-reference-01-execution-model.ipynb#repartition-and-coalesce)

*Hash partitioning by a skewed key puts the hot key's rows in one partition: the root of slow tasks.*

`clicks` has 1,000 rows, 900 for customer 1. Repartition it into 4 partitions by `customer_id`, then return `partition` and `rows`, ordered by `rows` descending then `partition`; include only non-empty partitions.

In [ ]:
clicks = spark.range(1000).select(F.when(F.col("id") < 900, 1).otherwise(F.col("id") % 9 + 2).alias("customer_id"))
clicks.groupBy("customer_id").count().orderBy("customer_id").show()

**Expected output**

| partition | rows |
| --- | --- |
| 1 | 933 |
| 0 | 45 |
| 3 | 22 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = (
    clicks.repartition(4, "customer_id")
    .groupBy(F.spark_partition_id().alias("partition")).agg(F.count("*").alias("rows"))
    .orderBy(F.desc("rows"), "partition")
)
result.show()

## Exercise 11: Partitions when reading many small files

**Difficulty:** Hard · about 15 min · **Practises:** [Partition count: reading, shuffling and AQE](pyspark-reference-01-execution-model.ipynb#partition-count-reading-shuffling-and-aqe)

*Small-file problems show up as partition counts that do not match the file count.*

Write `orders` as 12 Parquet files (`repartition(12)`) under `WORK_DIR`. Read it back with default settings and with `spark.sql.files.maxPartitionBytes` = `1k` and `spark.sql.files.openCostInBytes` = `1`. Return `setting` (`default`, `small_splits`) and `fewer_partitions_than_files` (true when the 12 files were packed into fewer than 12 partitions). Unset both settings afterwards.

In [ ]:
files_path = f"{WORK_DIR}/practice_small_files"
print(files_path)

**Expected output**

| setting | fewer_partitions_than_files |
| --- | --- |
| default | true |
| small_splits | false |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
orders.repartition(12).write.mode("overwrite").parquet(files_path)
default_parts = spark.read.parquet(files_path).rdd.getNumPartitions()
spark.conf.set("spark.sql.files.maxPartitionBytes", "1k")
spark.conf.set("spark.sql.files.openCostInBytes", "1")
small_parts = spark.read.parquet(files_path).rdd.getNumPartitions()
spark.conf.unset("spark.sql.files.maxPartitionBytes")
spark.conf.unset("spark.sql.files.openCostInBytes")
result = spark.createDataFrame([("default", default_parts < 12), ("small_splits", small_parts < 12)],
                               "setting string, fewer_partitions_than_files boolean")
result.show()

## Exercise 12: Count jobs and stages of an action

**Difficulty:** Hard · about 15 min · **Practises:** [How an action becomes jobs, stages and tasks](pyspark-reference-01-execution-model.ipynb#how-an-action-becomes-jobs-stages-and-tasks)

*Reading jobs and stages is how you connect code to the Spark UI.*

Run `orders.groupBy('customer_id').count().collect()` inside job group `practice-jobs` with AQE disabled. Return one row with `jobs` (jobs in the group) and `stages` (total stages across those jobs, including skipped ones). Re-enable AQE afterwards.

In [ ]:
tracker = spark.sparkContext.statusTracker()

**Expected output**

| jobs | stages |
| --- | --- |
| 1 | 2 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.sparkContext.setJobGroup("practice-jobs", "practice")
orders.groupBy("customer_id").count().collect()
job_ids = tracker.getJobIdsForGroup("practice-jobs")
stages = sum(len(tracker.getJobInfo(j).stageIds) for j in job_ids)
spark.sparkContext.setJobGroup("other", "")
spark.conf.set("spark.sql.adaptive.enabled", "true")
result = spark.createDataFrame([(len(job_ids), stages)], "jobs int, stages int")
result.show()

**Next:** look anything up in [PySpark reference 1: execution model](pyspark-reference-01-execution-model.ipynb), or test yourself across topics with the [PySpark coding challenges](pyspark-coding-challenges.ipynb).